Restaurant Recommendation System

In [35]:
# Load required libraries

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv('Dataset .csv')

df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [37]:
# data preprocessing

df['Cuisines'] = df['Cuisines'].fillna('Unknown')

data = df[[
    "Restaurant Name",
    "Cuisines",
    "City",
    "Price range",
    "Aggregate rating"
]].copy()

data.head()

,Restaurant Name,Cuisines,City,Price range,Aggregate rating
0,Le Petit Souffle,"French, Japanese, Desserts",Makati City,3,4.8
1,Izakaya Kikufuji,Japanese,Makati City,3,4.5
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",Mandaluyong City,4,4.4
3,Ooma,"Japanese, Sushi",Mandaluyong City,4,4.9
4,Sambo Kojin,"Japanese, Korean",Mandaluyong City,4,4.8


In [38]:
# Encoding categorical variables

encoded_data = pd.get_dummies(
    data,
    columns = ["Cuisines", "City", "Price range"],
    drop_first = False
)

encoded_data.head()

,Restaurant Name,Aggregate rating,Cuisines_Afghani,"Cuisines_Afghani, Mughlai, Chinese","Cuisines_Afghani, North Indian","Cuisines_Afghani, North Indian, Pakistani, Arabian",Cuisines_African,"Cuisines_African, Portuguese",Cuisines_American,"Cuisines_American, Asian, Burger",...,City_Waterloo,City_Weirton,City_Wellington City,City_Winchester Bay,City_Yorkton,City_��stanbul,Price range_1,Price range_2,Price range_3,Price range_4
0,Le Petit Souffle,4.8,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,Izakaya Kikufuji,4.5,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,Heat - Edsa Shangri-La,4.4,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,Ooma,4.9,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,Sambo Kojin,4.8,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [39]:
# User preferences

user_preference = {
    "Cuisines": "North Indian",
    "City": "New Delhi",
    "Price range": 3
}


In [40]:
# Feature alignment

feature_columns = encoded_data.drop(
    columns = ["Restaurant Name", "Aggregate rating"]
).columns

restaurant_vectors = encoded_data[feature_columns].values

user_vector = np.zeros(len(feature_columns))

for i,column in enumerate(feature_columns):
    if column == f"Cuisines_{user_preference['Cuisines']}":
        user_vector[i] = 1
    if column == f"City_{user_preference['City']}":
        user_vector[i] = 1
    if column == f"Price range_{user_preference['Price range']}":
        user_vector[i] = 1

In [41]:
# Similarity Calculation

similarity_scores = cosine_similarity(
    [user_vector],
    restaurant_vectors
)[0]

data["Similarity Score"] = similarity_scores

In [42]:
# Generating recommendation

recommendations = data.sort_values(
    by = ["Similarity Score", "Aggregate rating"],
    ascending = False
)

recommendations.head(10)


,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Similarity Score
6656,Kopper Kadai,North Indian,New Delhi,3,4.8,1.0
3014,Zabardast Indian Kitchen,North Indian,New Delhi,3,4.7,1.0
6655,Band Baaja Baaraat,North Indian,New Delhi,3,4.6,1.0
3999,Rang De Basanti Urban Dhaba,North Indian,New Delhi,3,4.4,1.0
6143,Chor Bizarre,North Indian,New Delhi,3,4.4,1.0
7469,Punjab Grill,North Indian,New Delhi,3,4.3,1.0
6145,Havemore,North Indian,New Delhi,3,4.1,1.0
7431,Masala House,North Indian,New Delhi,3,4.0,1.0
5893,Themis Barbecue House,North Indian,New Delhi,3,3.8,1.0
6142,Veg Gulati,North Indian,New Delhi,3,3.8,1.0


In [43]:
# Final recommended restaurants

final_recommendations = recommendations[[
    "Restaurant Name",
    "Cuisines",
    "City",
    "Price range",
    "Aggregate rating"
]].head(10)

final_recommendations


,Restaurant Name,Cuisines,City,Price range,Aggregate rating
6656,Kopper Kadai,North Indian,New Delhi,3,4.8
3014,Zabardast Indian Kitchen,North Indian,New Delhi,3,4.7
6655,Band Baaja Baaraat,North Indian,New Delhi,3,4.6
3999,Rang De Basanti Urban Dhaba,North Indian,New Delhi,3,4.4
6143,Chor Bizarre,North Indian,New Delhi,3,4.4
7469,Punjab Grill,North Indian,New Delhi,3,4.3
6145,Havemore,North Indian,New Delhi,3,4.1
7431,Masala House,North Indian,New Delhi,3,4.0
5893,Themis Barbecue House,North Indian,New Delhi,3,3.8
6142,Veg Gulati,North Indian,New Delhi,3,3.8


A context-based restaurant recommendation system is developrd.
The system provides relevant restaurant suggestions based on preferenecs.